In [50]:
# Imports
from google import genai
from google.genai import types
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd
import os
import json
import time
import base64
import sys
sys.path.append('..')
import prompts
import utils

In [51]:
load_dotenv(override=True)

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")

gemini_client = genai.Client()

gemini_model = os.getenv("GEMINI_MODEL")

images_folder_path = os.getenv("IMG_PATH")
output_folder_path = os.getenv("BASIC_OUTPUT_FOLDER_PATH")

In [52]:
def gemini_nutritionist(image):
    try:

        response = gemini_client.models.generate_content(
            model=gemini_model,
            contents=[
                types.Part.from_bytes(
                    data=image,
                    mime_type='image/jpeg',
                ),
                prompts.UNIFIED_NUTRITION_PROMPT
            ],
            config=types.GenerateContentConfig(
                temperature=0.1,
                response_mime_type="application/json",
                max_output_tokens=1024,
            )
        )

        if not response or not getattr(response, "text", None):
            return {'success': False, 'error': 'Gemini response is empty or missing `.text`'}
        
        try:
            data = utils.parse_json(response.text)
        except Exception as e:
            return {'success': False, 'error': f"No valid JSON found in the response text: {str(e)}"}

        return {
            'success': True,
            'description': data.get('description'),
            'calories': float(data.get('calories', 0)),
            'proteins': float(data.get('proteins', 0)),
            'carbohydrates': float(data.get('carbohydrates', 0)),
            'fats': float(data.get('fats', 0)),
            'serving_size': float(data.get('serving_size', 0))
        }
    except Exception as e:
        return {'success': False, 'error': str(e)}

In [ ]:
def analyze_image(image_path, index):
    """Complete chained analysis for one image"""
    start_time = time.time()
    file_name = os.path.basename(image_path)

    with open(image_path, "rb") as f:
            image_byte = f.read()

    print(f"\n🔄 Processing {index}: {file_name}")

    print("  Gemini Nutritionist...")
    response = gemini_nutritionist(image_byte)

    if not response['success']:
        return {'success': False, 'error': f"Gemini failed: {response['error']}", 'index': index}

    print(f"    → {response['success']}")
    if not response.get("description"):
        return {'success': False, 'error': "Gemini returned empty description", 'index': index}

    total_time = time.time() - start_time
    time.sleep(2)
    print(f"  ✅ Complete! {response['description']} in {total_time:.1f}s")

    return {
        'success': True,
        'id': index,
        'file_name': file_name,
        'response': response,
        'processing_time': total_time
    }

In [ ]:
def process_dataset(file_name, folder_path, start=1, end=None):
    """Process images with chained analysis"""

    total_images = utils.count_images(folder_path)
    if end is None:
        end = total_images
    end = min(end, total_images)
    
    print(f"🚀 Processing images {start} to {end} ({end-start+1} total)")
    
    results = []
    successful = 0
    
    for i in range(start, end + 1):
        image_path = os.path.join(folder_path, f"{i}.jpg")
        result = analyze_image(image_path, i)
        results.append(result)
        
        if result['success']:
            successful += 1
    
    print(f"\n🎉 Completed! {successful}/{len(results)} successful")
    
    # Save results
    output_file = f"{file_name}.json"
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    
    print(f"📁 Results saved to: {output_file}")
    return results

In [59]:
def export_to_excel(file_name):
    """Export results to Excel"""
    with open(f"{file_name}.json", 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Prepare final results
    final_results = []
    for item in data:
        if item['success']:
            nutrition = item['response']
            final_results.append({
                'id': item['index'],
                'description': nutrition['description'],
                'serving_size': nutrition['serving_size'],
                'calories': nutrition['calories'],
                'proteins': nutrition['proteins'],
                'carbohydrates': nutrition['carbohydrates'],
                'fats': nutrition['fats']
            })
    
    # Create DataFrame and export
    df = pd.DataFrame(final_results)
    output_path = os.path.join(output_folder_path, f"{file_name}.xlsx")
    df.to_excel(output_path, index=False)
    
    print(f"✅ Excel exported: {output_path}")
    print(f"📊 {len(final_results)} successful analyses")
    
    return output_path

In [56]:
file_name = "nutria_gemini"

In [57]:
results = process_dataset(file_name, images_folder_path, start=1, end=utils.count_images(images_folder_path))
# results = process_chained_dataset(file_name, images_folder_path, start=1, end=2)

🚀 Processing images 1 to 50 (50 total)
Sending image: /Users/martinhachiya/dev/datasets/nutria_backbone/food_images/1.jpg

🔄 Processing 1: 1.jpg
  Gemini Nutritionist...
    → True
  ✅ Complete! Vaso de cerveza clara. kcal in 4.0s
Sending image: /Users/martinhachiya/dev/datasets/nutria_backbone/food_images/2.jpg

🔄 Processing 2: 2.jpg
  Gemini Nutritionist...
    → True
  ✅ Complete! Pasta penne con salsa de tomate y albahaca kcal in 2.9s
Sending image: /Users/martinhachiya/dev/datasets/nutria_backbone/food_images/3.jpg

🔄 Processing 3: 3.jpg
  Gemini Nutritionist...
    → True
  ✅ Complete! El plato contiene un croissant integral con semillas, huevos revueltos, salmón ahumado y dos tomates cherry. kcal in 2.6s
Sending image: /Users/martinhachiya/dev/datasets/nutria_backbone/food_images/4.jpg

🔄 Processing 4: 4.jpg
  Gemini Nutritionist...
    → True
  ✅ Complete! Dos rebanadas de lomo de cerdo envuelto en tocino con salsa y papas fritas. kcal in 3.2s
Sending image: /Users/martinhachiy

In [60]:
excel_path = export_to_excel(file_name)
print(f"\n🎯 Done! Check: {excel_path}")

✅ Excel exported: /Users/martinhachiya/dev/datasets/nutria_backbone/llm_response/basic/nutria_gemini.xlsx
📊 50 successful analyses

🎯 Done! Check: /Users/martinhachiya/dev/datasets/nutria_backbone/llm_response/basic/nutria_gemini.xlsx
